# Mriya Report — capture to Drive

Runs on Google Colab, writes straight to Google Drive. Nothing passes through your own
connection.

**Order matters: indexes first, media second.** An index cannot be reconstructed after a channel
goes down — if capture is interrupted, an index in hand says exactly what the missing remainder
was. Enumeration takes minutes; media takes hours.

Every media file is hashed **before any processing**, and the hash goes in the manifest. That is
what makes a transcript checkable later.

Target Drive folder: `Mriya Report Transcripts — audio`


In [ ]:
#@title 1 · Mount Drive and install tools
from google.colab import drive
drive.mount('/content/drive')

!pip -q install yt-dlp requests
!apt-get -qq install -y ffmpeg > /dev/null
import os, pathlib

ROOT = '/content/drive/MyDrive/Mriya Report Transcripts — audio'
for sub in ('', 'audio-2022', 'audio-2023', 'audio-2024', 'index', 'sitetext', 'logs'):
    pathlib.Path(os.path.join(ROOT, sub)).mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT)
print(os.listdir(ROOT))

## 2 · Index the podcast feed

Writes `index/feed-<date>.xml` (the raw feed is itself an artifact — keep it and hash it) and
`index/MANIFEST-feed.csv` with enclosure URLs and GUIDs, which is what the metadata-only manifest
in the repo is missing.


In [ ]:
#@title 2 · Feed index
import csv, hashlib, html, os, re, requests, datetime
import xml.etree.ElementTree as ET
from email.utils import parsedate_to_datetime

FEED = 'https://anchor.fm/s/ace6a3f0/podcast/rss'
NS = {'itunes':'http://www.itunes.com/dtds/podcast-1.0.dtd'}
today = datetime.date.today().isoformat()

raw = requests.get(FEED, timeout=120, headers={'User-Agent':'prime-rogue-transcripts/1.0'}).content
xml_path = f'{ROOT}/index/feed-{today}.xml'
open(xml_path,'wb').write(raw)
print(f'feed saved  {len(raw):,} bytes  sha256={hashlib.sha256(raw).hexdigest()}')

def txt(el):
    return html.unescape(''.join(el.itertext())).strip() if el is not None else ''

ch = ET.fromstring(raw).find('channel')
show = txt(ch.find('title'))
items = []
for it in ch.findall('item'):
    enc = it.find('enclosure')
    pd = txt(it.find('pubDate'))
    try: iso = parsedate_to_datetime(pd).date().isoformat()
    except Exception: iso = pd
    desc = re.sub(r'\s+',' ', re.sub(r'<[^>]+>',' ', txt(it.find('description'))))
    items.append(dict(title=txt(it.find('title')), guid=txt(it.find('guid')), published=iso,
                      duration=txt(it.find('itunes:duration', NS)),
                      url=enc.get('url') if enc is not None else '',
                      length=enc.get('length') if enc is not None else '', desc=desc))
items.sort(key=lambda i: str(i['published']))

F = ['media_id','title','show','platform','source_id','url','guid','published','duration',
     'enclosure_bytes','handles_in_metadata','acquisition_status','media_sha256','captured_at',
     'capture_method','transcript_state','live_at_source','last_checked','description']
counters = {}
with open(f'{ROOT}/index/MANIFEST-feed.csv','w',newline='',encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=F); w.writeheader()
    for i in items:
        yr = str(i['published'])[:4] or 'ALL'
        counters[yr] = counters.get(yr,0)+1
        w.writerow({'media_id':f"MR-{yr}-{counters[yr]:04d}", 'title':i['title'], 'show':show,
            'platform':'anchor/rss','source_id':'S-T01','url':i['url'],'guid':i['guid'],
            'published':i['published'],'duration':i['duration'],'enclosure_bytes':i['length'],
            'handles_in_metadata':' '.join(sorted(set(re.findall(r'@[A-Za-z0-9_]{2,15}',
                i['title']+' '+i['desc'])))),
            'acquisition_status':'listed','media_sha256':'','captured_at':'',
            'capture_method':'colab feed index','transcript_state':'',
            'live_at_source':'yes','last_checked':today,'description':i['desc']})
print(f'{len(items)} items;  by year:', dict(sorted(counters.items())))
print('oldest:', items[0]['published'], '|', items[0]['title'][:70])

## 3 · Index the YouTube channel — all 566 videos

This is the step that closes the gap. The channel listing captured by hand starts at
**12 August 2022**; the channel has been broadcasting since the day before the full-scale
invasion. This enumerates the lot, with real upload dates rather than "x years ago".


In [ ]:
#@title 3 · Channel index
import json, subprocess, csv, datetime
today = datetime.date.today().isoformat()
CHANNEL = 'https://www.youtube.com/channel/UCuuRSmrJXRUUMW-0WQ_JIhw'

# --flat-playlist keeps this to one metadata pass, no media
out = subprocess.run(['yt-dlp','--flat-playlist','-J','--ignore-errors',CHANNEL+'/videos'],
                     capture_output=True, text=True)
data = json.loads(out.stdout)
entries = [e for e in data.get('entries') or [] if e]
print(f'{len(entries)} entries from the flat pass')

# the flat pass omits upload_date; fetch per-video metadata (no media) to get real dates
rows = []
for n, e in enumerate(entries, 1):
    vid = e.get('id')
    try:
        m = json.loads(subprocess.run(
            ['yt-dlp','-J','--skip-download','--no-warnings',f'https://www.youtube.com/watch?v={vid}'],
            capture_output=True, text=True).stdout)
    except Exception:
        m = {}
    ud = m.get('upload_date') or ''
    rows.append(dict(video_id=vid, title=m.get('title') or e.get('title') or '',
        published=(f'{ud[:4]}-{ud[4:6]}-{ud[6:8]}' if len(ud)==8 else ''),
        duration_s=m.get('duration') or e.get('duration') or '',
        views=m.get('view_count') or '', url=f'https://www.youtube.com/watch?v={vid}',
        description=(m.get('description') or '').replace('\n',' ')[:2000]))
    if n % 25 == 0: print(f'  {n}/{len(entries)}')

rows.sort(key=lambda r: (r['published'] or '9999'))
def hms(s):
    s = int(s or 0); return f'{s//3600:02d}:{s%3600//60:02d}:{s%60:02d}'
cnt = {}
F = ['media_id','video_id','title','platform','source_id','url','published','duration',
     'views_at_capture','acquisition_status','media_sha256','captured_at','capture_method',
     'live_at_source','last_checked','description']
with open(f'{ROOT}/index/YT-INDEX-full.csv','w',newline='',encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=F); w.writeheader()
    for r in rows:
        yr = (r['published'] or 'unknown')[:4]; cnt[yr] = cnt.get(yr,0)+1
        w.writerow({'media_id':f"MR-YT-{yr}-{cnt[yr]:04d}", 'video_id':r['video_id'],
            'title':r['title'],'platform':'youtube','source_id':'S-T04','url':r['url'],
            'published':r['published'],'duration':hms(r['duration_s']),
            'views_at_capture':r['views'],'acquisition_status':'listed','media_sha256':'',
            'captured_at':'','capture_method':'colab yt-dlp index','live_at_source':'yes',
            'last_checked':today,'description':r['description']})
total = sum(int(r['duration_s'] or 0) for r in rows)
print(f"\n{len(rows)} videos;  by year:", dict(sorted(cnt.items())))
print(f"total runtime {total//3600:,} h {total%3600//60} m")
print("oldest:", rows[0]['published'], '|', rows[0]['title'][:70])

## 4 · Capture the site text

Small, and it goes when the channel goes. About pages, channel description, show description.


In [ ]:
#@title 4 · Site text
import json, subprocess, requests, datetime
today = datetime.date.today().isoformat()

meta = json.loads(subprocess.run(['yt-dlp','-J','--skip-download','--playlist-items','0',CHANNEL],
                                 capture_output=True, text=True).stdout or '{}')
open(f'{ROOT}/sitetext/youtube-channel-about-{today}.json','w',encoding='utf-8').write(
    json.dumps({k:meta.get(k) for k in
    ('id','channel','uploader','description','channel_follower_count','tags')}, indent=2))

for name, url in [('mriyareport.org','https://mriyareport.org/'),
                  ('mriyareport.net','https://mriyareport.net/')]:
    try:
        r = requests.get(url, timeout=60, headers={'User-Agent':'Mozilla/5.0'})
        open(f'{ROOT}/sitetext/{name}-{today}.html','wb').write(r.content)
        print(f'{name}: {r.status_code}, {len(r.content):,} bytes')
    except Exception as ex:
        print(f'{name}: FAILED — {ex}  (record this in removals.csv if it is gone)')
print('site text saved')

## 5 · Download media

`YEAR` and `SOURCE` pick the slice. Each file is hashed **before** anything else touches it, and
the hash is appended to `SHA256SUMS`. Re-running skips what is already present, so this is safe to
interrupt and resume — which matters, because 1,264 hours of 2022 video is a long run.

Audio only, best available, no re-encode unless the source has no audio-only stream.


In [ ]:
#@title 5 · Media capture
YEAR   = '2022'      #@param ['2022','2023','2024']
SOURCE = 'youtube'   #@param ['youtube','feed']
LIMIT  = 0           #@param {type:'integer'}

import csv, hashlib, os, subprocess, datetime
dest = f'{ROOT}/audio-{YEAR}'; os.makedirs(dest, exist_ok=True)
idx = f'{ROOT}/index/' + ('YT-INDEX-full.csv' if SOURCE=='youtube' else 'MANIFEST-feed.csv')
rows = [r for r in csv.DictReader(open(idx, encoding='utf-8'))
        if (r.get('published') or '').startswith(YEAR)]
if LIMIT: rows = rows[:LIMIT]
print(f'{len(rows)} items queued from {os.path.basename(idx)}')

def sha256(p):
    h = hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda: f.read(1<<20), b''): h.update(b)
    return h.hexdigest()

sums = f'{ROOT}/SHA256SUMS'
done = set()
if os.path.exists(sums):
    done = {l.split()[1] for l in open(sums) if len(l.split())>1 and not l.startswith('#')}

for n, r in enumerate(rows, 1):
    mid = r['media_id']
    if mid in done:
        print(f'[{n}/{len(rows)}] {mid} already hashed, skipping'); continue
    print(f'[{n}/{len(rows)}] {mid}  {r.get("duration","")}  {r["title"][:60]}')
    if SOURCE == 'youtube':
        cmd = ['yt-dlp','-f','bestaudio/best','--no-warnings','-o',f'{dest}/{mid}.%(ext)s', r['url']]
    else:
        cmd = ['curl','-sS','-L','--retry','3','-o',f'{dest}/{mid}.mp3', r['url']]
    rc = subprocess.run(cmd).returncode
    if rc != 0:
        print(f'   FAILED rc={rc} — if the item is gone, record it in removals.csv'); continue
    got = [f for f in os.listdir(dest) if f.startswith(mid+'.')]
    if not got: print('   no file produced'); continue
    p = os.path.join(dest, got[0]); h = sha256(p)
    with open(sums,'a') as f:
        f.write(f'{h}  {mid}  {got[0]}\n')
    print(f'   {os.path.getsize(p):,} bytes  sha256={h[:16]}…')

print('\ndone. Fold the hashes back into the manifest media_sha256 column.')

## 6 · Fold hashes into the manifest

Run after any capture pass, so the manifest and `SHA256SUMS` agree.


In [ ]:
#@title 6 · Reconcile
import csv, os
sums = {}
p = f'{ROOT}/SHA256SUMS'
if os.path.exists(p):
    for l in open(p):
        parts = l.split()
        if len(parts) >= 2 and not l.startswith('#'): sums[parts[1]] = parts[0]

for name in ('MANIFEST-feed.csv','YT-INDEX-full.csv'):
    fp = f'{ROOT}/index/{name}'
    if not os.path.exists(fp): continue
    rows = list(csv.DictReader(open(fp, encoding='utf-8')))
    flds = rows[0].keys() if rows else []
    n = 0
    for r in rows:
        if r['media_id'] in sums and not r.get('media_sha256'):
            r['media_sha256'] = sums[r['media_id']]
            r['acquisition_status'] = 'acquired'; n += 1
    with open(fp,'w',newline='',encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(flds)); w.writeheader(); w.writerows(rows)
    print(f'{name}: {n} rows updated, {sum(1 for r in rows if r.get("media_sha256"))} acquired total')